# 02.6 - Backtracking

## What is backtracking?
A systematic way to try **all possibilities** by building a solution step by step, and **undoing** (backtracking) a step when it can't lead to a valid answer.

It's DFS over a **decision tree**:
```
                      []
           /           |          \
         [1]          [2]         [3]        choose
        /   \        /   \       /   \
     [1,2] [1,3]  [2,1] [2,3]  [3,1] [3,2]   explore
       |     |      |     |      |     |
    [1,2,3] ...                               record, then un-choose
```

## The template (memorise it)
```python
def backtrack(state):
    if is_solution(state):
        record(state)
        return
    for choice in choices(state):
        if is_valid(choice):
            make(choice)          # choose
            backtrack(state)      # explore
            undo(choice)          # un-choose
```
**Pruning** = skip choices that can't work → huge speed-ups.

## Where is it used?
| Real system | Use |
|---|---|
| **Sudoku / crossword / puzzle solvers** | Fill cells, undo on conflict |
| **Constraint solvers** (timetables, exam scheduling, rostering) | Assign values satisfying all constraints |
| **Regex engines** | Try alternative matches |
| **Game AI** (chess, tic-tac-toe) | Explore moves (minimax) |
| **Compilers / SAT solvers / chip design** | Search for satisfying assignments |
| **Password / combination testing (security research)** | Enumerate candidates |

---
## Solved problems

### 1. Letter Combinations of a Phone Number - LC #17 (Medium) Digits 2-9 → all letter combinations (old phone keypads / T9).

In [1]:
def letter_combinations(digits):
    if not digits:
        return []
    keypad = {"2": "abc", "3": "def", "4": "ghi", "5": "jkl",
              "6": "mno", "7": "pqrs", "8": "tuv", "9": "wxyz"}
    result = []
    def backtrack(i, path):
        if i == len(digits):
            result.append("".join(path))
            return
        for ch in keypad[digits[i]]:
            path.append(ch)
            backtrack(i + 1, path)
            path.pop()
    backtrack(0, [])
    return result

assert letter_combinations("23") == ["ad", "ae", "af", "bd", "be", "bf", "cd", "ce", "cf"]
assert letter_combinations("") == []
print("Letter Combinations ")

Letter Combinations 


### 2. Subsets II (with duplicates) - LC #90 (Medium)
All unique subsets when the input has duplicates.

**Idea:** Sort, then at each level skip a number equal to the previous one (it would create the same subset).

In [2]:
def subsets_with_dup(nums):
    nums.sort()
    result, path = [], []
    def backtrack(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:     # skip duplicate choices at this level
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

assert sorted(subsets_with_dup([1, 2, 2])) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
print("Subsets II ")

Subsets II 


### 3. Combination Sum II - LC #40 (Medium)
Each number used at most once; unique combinations summing to target.

In [3]:
def combination_sum2(candidates, target):
    candidates.sort()
    result, path = [], []
    def backtrack(start, remaining):
        if remaining == 0:
            result.append(path[:])
            return
        for i in range(start, len(candidates)):
            if i > start and candidates[i] == candidates[i - 1]:
                continue
            if candidates[i] > remaining:
                break                                  # prune: sorted, so the rest are too big
            path.append(candidates[i])
            backtrack(i + 1, remaining - candidates[i])
            path.pop()
    backtrack(0, target)
    return result

assert combination_sum2([10, 1, 2, 7, 6, 1, 5], 8) == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
print("Combination Sum II ")

Combination Sum II 


### 4. Word Search - LC #79 (Medium) Does `word` exist in the grid (adjacent cells, no reuse)?

In [4]:
def exist(board, word):
    rows, cols = len(board), len(board[0])
    def backtrack(r, c, i):
        if i == len(word):
            return True
        if r < 0 or c < 0 or r >= rows or c >= cols or board[r][c] != word[i]:
            return False
        saved, board[r][c] = board[r][c], "#"           # choose (mark used)
        found = any(backtrack(r + dr, c + dc, i + 1)
                    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)))
        board[r][c] = saved                              # un-choose
        return found
    return any(backtrack(r, c, 0) for r in range(rows) for c in range(cols))

board = [["A", "B", "C", "E"], ["S", "F", "C", "S"], ["A", "D", "E", "E"]]
assert exist(board, "ABCCED") and exist(board, "SEE") and not exist(board, "ABCB")
print("Word Search ")

Word Search 


### 5. Palindrome Partitioning - LC #131 (Medium)
All ways to split `s` so every part is a palindrome.

In [5]:
def partition(s):
    result, path = [], []
    def backtrack(start):
        if start == len(s):
            result.append(path[:])
            return
        for end in range(start + 1, len(s) + 1):
            piece = s[start:end]
            if piece == piece[::-1]:                     # prune non-palindromes
                path.append(piece)
                backtrack(end)
                path.pop()
    backtrack(0)
    return result

assert partition("aab") == [["a", "a", "b"], ["aa", "b"]]
assert partition("a") == [["a"]]
print("Palindrome Partitioning ")

Palindrome Partitioning 


### 6. N-Queens - LC #51 (Hard) the classic
Place `n` queens on an n×n board so none attack each other. Return all boards.

**Idea:** One queen per row. Track used columns and both diagonals in sets (cells on the same diagonal share `r - c` or `r + c`).

In [6]:
def solve_n_queens(n):
    result = []
    cols, diag1, diag2 = set(), set(), set()       # diag1: r - c, diag2: r + c
    board = [["."] * n for _ in range(n)]
    def backtrack(r):
        if r == n:
            result.append(["".join(row) for row in board])
            return
        for c in range(n):
            if c in cols or (r - c) in diag1 or (r + c) in diag2:
                continue                            # attacked -> prune
            cols.add(c); diag1.add(r - c); diag2.add(r + c); board[r][c] = "Q"
            backtrack(r + 1)
            cols.remove(c); diag1.remove(r - c); diag2.remove(r + c); board[r][c] = "."
    backtrack(0)
    return result

solutions = solve_n_queens(4)
assert solutions == [[".Q..", "...Q", "Q...", "..Q."], ["..Q.", "Q...", "...Q", ".Q.."]]
assert len(solve_n_queens(8)) == 92
print("\n".join(solutions[0]))
print("N-Queens ")

.Q..
...Q
Q...
..Q.
N-Queens 


### 7. Sudoku Solver - LC #37 (Hard) - real puzzle solver
Fill the board in place.

In [7]:
def solve_sudoku(board):
    rows = [set() for _ in range(9)]
    cols = [set() for _ in range(9)]
    boxes = [set() for _ in range(9)]
    empty = []
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == ".":
                empty.append((r, c))
            else:
                rows[r].add(v); cols[c].add(v); boxes[(r // 3) * 3 + c // 3].add(v)

    def backtrack(k):
        if k == len(empty):
            return True
        r, c = empty[k]
        b = (r // 3) * 3 + c // 3
        for v in "123456789":
            if v not in rows[r] and v not in cols[c] and v not in boxes[b]:
                board[r][c] = v
                rows[r].add(v); cols[c].add(v); boxes[b].add(v)
                if backtrack(k + 1):
                    return True
                rows[r].remove(v); cols[c].remove(v); boxes[b].remove(v)
        board[r][c] = "."
        return False

    backtrack(0)

puzzle = ["53..7....", "6..195...", ".98....6.", "8...6...3", "4..8.3..1",
          "7...2...6", ".6....28.", "...419..5", "....8..79"]
board = [list(row) for row in puzzle]
solve_sudoku(board)
for row in board:
    print(" ".join(row))
assert "".join(board[0]) == "534678912" and all("." not in row for row in board)
print("Sudoku Solver ")

5 3 4 6 7 8 9 1 2
6 7 2 1 9 5 3 4 8
1 9 8 3 4 2 5 6 7
8 5 9 7 6 1 4 2 3
4 2 6 8 5 3 7 9 1
7 1 3 9 2 4 8 5 6
9 6 1 5 3 7 2 8 4
2 8 7 4 1 9 6 3 5
3 4 5 2 8 6 1 7 9
Sudoku Solver 


## More practice on LeetCode
Medium: Combinations (#77), Permutations II (#47), Restore IP Addresses (#93), Matchsticks to Square (#473), Partition to K Equal Sum Subsets (#698)
Hard: N-Queens II (#52), Word Search II (#212 - in 01.8)

Also in `03-Interview Patterns/03.1`: Subsets, Permutations, Combination Sum.

## Key takeaways
- "Find **all** …" / "generate every …" → backtracking.
- Template: **choose → explore → un-choose**; copy `path[:]` when recording.
- **Prune early** (sorted input + `break`, validity sets) - that's what makes it fast enough.
- Complexity is exponential (O(2ⁿ), O(n!)) - check that n is small (≤ ~15-20).

**Next: `02.7-Dynamic-Programming.ipynb`**